In [7]:
# Make a simple svm, linear regression and random forest regression model to predict y from x
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
X_train, X_test, y_train, y_test = train_test_split(x.reshape(-1, 1), y, test_size=0.1, random_state=42)
models = {
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(),
    'LinearRegression': LinearRegression()
}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f'{name} MSE: {mse}')


SVR MSE: 2374234354.79043
RandomForest MSE: 48398912927.08367
LinearRegression MSE: 2485924163.51955


In [8]:
# Example of how to use the model to predict duration for a new input size
new_input_size = 500000000  # 500 MB
for name, model in models.items():
    predicted_duration = model.predict([[new_input_size]])
    print(f'{name} predicted duration for input size {new_input_size} bytes: {predicted_duration[0]} seconds')


SVR predicted duration for input size 500000000 bytes: 14411.946110872112 seconds
RandomForest predicted duration for input size 500000000 bytes: 21985.805142857145 seconds
LinearRegression predicted duration for input size 500000000 bytes: 77744.10504058322 seconds


## Define Clusters and Workflows

## Load Cluster Node Profiles

In [ ]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

In [ ]:
nodes = ['aws-metal-c5n', 'aws-metal-z1d', 'gpgnode-16', 'gpgnode-22', 'tu-06']  #, 'hu-c40']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

{'ram': 128, 'cores': 32, 'io_read': 3500.0, 'io_write': 4717.0, 'sysbench': 601.07, 'z7b': 3014.0, 'bench_time': 0.07366666666666667}


In [12]:
# Imports
import pandas as pd

## Predict from TU-06

In [ ]:
# tu-06
tu_06_profile = node_profiles['tu-06']
tu_06_profiling_dir = 'tu-06/'
workflows = ['rnaseq', 'chipseq']
rnaseq_samples = ['sample_b', 'sample_c', 'sample_d']
chipseq_samples = ['sample_e', 'sample_f', 'sample_g', 'sample_h']

for workflow, samples in zip(workflows, [rnaseq_samples, chipseq_samples]):
    for sample in samples:
        sample_dir = f'{tu_06_profiling_dir}{workflow}/{sample}/'
        input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
        trace_df = pd.read_csv(sample_dir + 'trace.csv')
        print(input_df)
        print(trace_df)
        break
    break

# todo you are here @Kath

                                                  name  \
0    NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   
1    NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   
2    NFCORE_RNASEQ:RNASEQ:FASTQ_QC_TRIM_FILTER_SETS...   
3    NFCORE_RNASEQ:RNASEQ:FASTQ_QC_TRIM_FILTER_SETS...   
4    NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...   
..                                                 ...   
147                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
148                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
149                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
150                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   
151                   NFCORE_RNASEQ:RNASEQ:MULTIQC (1)   

                                 hash  \
0    28da6992782f41ecefaa5f441691b1b7   
1    28da6992782f41ecefaa5f441691b1b7   
2    4288e0bab37927385f33dc39a298e8b2   
3    4288e0bab37927385f33dc39a298e8b2   
4    b9eb959b294ef9e098e5f2609f1882d2   
..                                ...   
147  efc1db4803a

In [2]:
# Group by name and show the different files
for name, group in input_df.groupby('name'):
    # print(group[['path', 'size']])
    print(str(group[['name']]) + ' has ' + str(len(group)) + ' files.')
    total_size = group['size'].sum()
    print('Total size: ' + str(total_size) + ' bytes.')

# Assert length of traces is equal to number of unique tasks in input
assert len(trace_df) == len(input_df['name'].unique())


                                                name
4  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI... has 1 files.
Total size: 3147288982 bytes.
                                                name
0  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...
1  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom... has 2 files.
Total size: 4063383142 bytes.
                                                 name
9   NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_...
10  NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_... has 2 files.
Total size: 3985567244 bytes.
                                                 name
17  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen...
18  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen... has 2 files.
Total size: 3437932337 bytes.
                                                 name
31  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...
32  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS... has 2 files.
Total size: 175770742 bytes.
                                   

In [3]:
# Make x the sum of input_df grouped by task name
x = input_df.groupby('name')['size'].sum().values
y = trace_df['duration'].values

# Assert lengths are equal
assert len(x) == len(y)

## Predict from HU-c40